In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train_fe_v3.csv')
test  = pd.read_csv('test_fe_v3.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print('Train:', train.shape, '/ Test:', test.shape)
print()
print('시술 유형 분포 (train)')
print(train['시술 유형'].value_counts())
print()
print('시술 유형 분포 (test)')
print(test['시술 유형'].value_counts())

In [ ]:
# feature_refinement 재적용
ZERO_IMP_COLS = [
    '배아생성_제로_플래그','배아생성_실패_플래그','난자수집_실패_플래그',
    '난자저장용_포함','배반포이식_여부','불임 원인 - 여성 요인',
    'IVF_배반포_조합','불임 원인 - 정자 면역학적 요인','high_resp_freezeall',
]
drop_cols = [c for c in ZERO_IMP_COLS if c in train.columns]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

for df in [train, test]:
    emb_n   = df['이식된 배아 수'].fillna(0)
    emb_log = df['이식된 배아 수_log'].fillna(0)
    age     = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    is_ivf  = (df['시술 유형'] == 'IVF').astype(int)
    is_tx   = df['이식배아_있음'].fillna(0)
    tx_day  = df['배아 이식 경과일'].replace(-1, np.nan).fillna(0)

    df['이식배아수_나이_교호']     = emb_n * age
    df['이식배아log_나이_교호']    = emb_log * age
    df['이식배아수_품질_교호']     = emb_n * quality
    df['이식배아log_품질_교호']    = emb_log * quality
    df['이식배아수_IVF_교호']      = emb_n * is_ivf
    df['이식성공_품질_교호']       = is_tx * quality
    df['고령자가난자_품질_교호']   = df['자가난자_고령_조합'].fillna(0) * quality
    df['고위험3중_이식배아_교호']  = df['고위험_3중_조합'].fillna(0) * emb_n
    df['이식경과일_이식배아_교호'] = tx_day * emb_n
    df['이식배아수_출산경험_교호'] = emb_n * df['출산경험_있음'].fillna(0)
    df['나이_저장비율_교호']       = age * df['저장_비율'].fillna(0)
    df['잔여배아_품질_교호']       = df['잔여배아_수'].fillna(0) * quality
    df['이식배아수_제곱']          = emb_n ** 2
    df['이식배아_총배아_곱']       = emb_n * df['총 생성 배아 수'].fillna(0)
    df['단일배아이식_여부']        = (emb_n == 1).astype(int)
    df['2개배아이식_여부']         = (emb_n == 2).astype(int)

print('피처 정제 완료 / Train:', train.shape)

In [ ]:
# IVF / DI 데이터 분리
# 분리
train_ivf = train[train['시술 유형'] == 'IVF'].reset_index(drop=True)
train_di  = train[train['시술 유형'] != 'IVF'].reset_index(drop=True)
test_ivf  = test[test['시술 유형'] == 'IVF'].reset_index(drop=True)
test_di   = test[test['시술 유형'] != 'IVF'].reset_index(drop=True)

print(f'Train IVF: {len(train_ivf)}건 / DI: {len(train_di)}건')
print(f'Test  IVF: {len(test_ivf)}건 / DI: {len(test_di)}건')
print()
print(f'IVF 타겟 분포: {train_ivf[TARGET].mean():.3f}')
print(f'DI  타겟 분포: {train_di[TARGET].mean():.3f}')

# DI 샘플 수 경고
if len(train_di) < 200:
    print(f'\n⚠️ DI 샘플 수 {len(train_di)}건 — 5-fold 대신 3-fold 사용 권장')

In [ ]:
# 피처 준비 - IVF / DI 각각
DROP_COLS = [TARGET, ID_COL]

# DI에서 의미 없는 IVF 전용 피처 제거
# (DI에서는 전부 0 또는 -1로 채워진 피처 → 분산 없어 노이즈)
IVF_ONLY_FEATURES = [
    '총 생성 배아 수', '이식된 배아 수', '저장된 배아 수',
    '미세주입된 난자 수', '미세주입에서 생성된 배아 수',
    '수집된 신선 난자 수', '혼합된 난자 수',
    '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수',
    '해동된 배아 수', '해동 난자 수',
    '총 생성 배아 수_log', '이식된 배아 수_log',
    '수집된 신선 난자 수_log', '혼합된 난자 수_log', '저장된 배아 수_log',
    '배아품질_종합점수', '수정_효율', 'ICSI_수정_효율', '이식_효율',
    '저장_비율', '난자_배아_전환율', '난자수_log',
    '배양_기간_일', '채취_이식_총기간', '채취_혼합_간격', '해동_이식_간격',
    '배아 이식 경과일', '난자 채취 경과일', '난자 혼합 경과일', '배아 해동 경과일',
    '이식배아_있음', '배아생성_있음', '배아저장_있음', 'ICSI시술_있음', '해동배아_있음',
    '이식배아_구간', '잔여배아_수', '배양기간_D5추정', '이식_단계_범주',
    '이식배아수_나이_교호', '이식배아log_나이_교호',
    '이식배아수_품질_교호', '이식배아log_품질_교호',
    '이식배아수_IVF_교호', '이식성공_품질_교호',
    '고령자가난자_품질_교호', '고위험3중_이식배아_교호',
    '이식경과일_이식배아_교호', '이식배아수_출산경험_교호',
    '나이_저장비율_교호', '잔여배아_품질_교호',
    '이식배아수_제곱', '이식배아_총배아_곱',
    '단일배아이식_여부', '2개배아이식_여부',
    'poor_responder', 'high_responder', 'freeze_all_전략',
    '난소반응_그룹', 'ICSI_여부', 'FER_여부', 'AH_여부',
    'ICSI_배반포_조합', '기술집약도_점수', 'RIF_플래그',
    'RIF_AH_조합', 'RIF_PGT_조합', 'PGT_사용_여부',
    '고령_PGT_조합', 'PGT_단일이식_조합',
    '나이_배아품질_상호작용', 'poor_resp_수정효율',
]
IVF_ONLY_FEATURES = [c for c in IVF_ONLY_FEATURES if c in train.columns]

# IVF 피처: 전체
ivf_feat_cols = [c for c in train_ivf.columns if c not in DROP_COLS]

# DI 피처: IVF 전용 제거
di_feat_cols = [
    c for c in train_di.columns
    if c not in DROP_COLS and c not in IVF_ONLY_FEATURES
]

print(f'IVF 피처 수: {len(ivf_feat_cols)}개')
print(f'DI  피처 수: {len(di_feat_cols)}개')

In [ ]:
def prepare_features(train_df, test_df, feat_cols):
    X      = train_df[feat_cols].copy()
    y      = train_df[TARGET].copy()
    X_test = test_df[feat_cols].copy()

    cat_features = [
        c for c in feat_cols
        if X[c].dtype == 'object' or pd.api.types.is_string_dtype(X[c])
    ]
    for col in cat_features:
        X[col]      = X[col].astype(str).fillna('missing')
        X_test[col] = X_test[col].astype(str).fillna('missing')

    cat_indices = [X.columns.get_loc(c) for c in cat_features]

    X_lgb      = X.copy()
    X_test_lgb = X_test.copy()
    for col in cat_features:
        X_lgb[col]      = X_lgb[col].astype('category')
        X_test_lgb[col] = X_test_lgb[col].astype('category')

    neg = (y == 0).sum()
    pos = (y == 1).sum()
    spw = neg / pos

    return X, y, X_test, cat_features, cat_indices, X_lgb, X_test_lgb, spw

# IVF
X_ivf, y_ivf, X_test_ivf, cat_ivf, cat_idx_ivf, X_lgb_ivf, X_test_lgb_ivf, spw_ivf = \
    prepare_features(train_ivf, test_ivf, ivf_feat_cols)

# DI
X_di, y_di, X_test_di, cat_di, cat_idx_di, X_lgb_di, X_test_lgb_di, spw_di = \
    prepare_features(train_di, test_di, di_feat_cols)

print(f'IVF — X: {X_ivf.shape}, spw: {spw_ivf:.2f}')
print(f'DI  — X: {X_di.shape},  spw: {spw_di:.2f}')

In [ ]:
# IVF 모델 - CatBoost + Light GBM 앙상블
SEED = 42
N_IVF = 5  # IVF는 샘플 많으므로 5-fold
skf_ivf = StratifiedKFold(n_splits=N_IVF, shuffle=True, random_state=SEED)

# CatBoost Optuna 최적 파라미터
cat_params = {
    'iterations': 794, 'learning_rate': 0.030094391673729362,
    'depth': 7, 'l2_leaf_reg': 7.433949857398995,
    'bagging_temperature': 0.3980358270898108,
    'random_strength': 1.3075975210328405, 'border_count': 108,
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'scale_pos_weight': spw_ivf, 'random_seed': SEED,
    'early_stopping_rounds': 50, 'verbose': False, 'use_best_model': True,
}

# LightGBM Optuna 최적 파라미터 (model_lgb_optuna에서 나온 값으로 교체)
lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 22, 'max_depth': 7, 'learning_rate': 0.02030926523583015,
    'n_estimators': 956, 'subsample': 0.9916893928795039,
    'colsample_bytree': 0.8760988294276582,
    'reg_alpha': 0.5158539052029842, 'reg_lambda': 0.05557192411355649,
    'min_child_samples': 98,
    'scale_pos_weight': spw_ivf, 'random_state': SEED, 'verbose': -1,
}

oof_cat_ivf  = np.zeros(len(X_ivf))
oof_lgb_ivf  = np.zeros(len(X_ivf))
test_cat_ivf = np.zeros(len(X_test_ivf))
test_lgb_ivf = np.zeros(len(X_test_ivf))

for fold, (tr_idx, val_idx) in enumerate(skf_ivf.split(X_ivf, y_ivf)):
    X_tr  = X_ivf.iloc[tr_idx].reset_index(drop=True)
    X_val = X_ivf.iloc[val_idx].reset_index(drop=True)
    y_tr  = y_ivf.iloc[tr_idx].reset_index(drop=True)
    y_val = y_ivf.iloc[val_idx].reset_index(drop=True)

    # CatBoost
    cb = CatBoostClassifier(**cat_params)
    cb.fit(Pool(X_tr, y_tr, cat_features=cat_idx_ivf),
           eval_set=Pool(X_val, y_val, cat_features=cat_idx_ivf))
    oof_cat_ivf[val_idx]  = cb.predict_proba(Pool(X_val, cat_features=cat_idx_ivf))[:, 1]
    test_cat_ivf         += cb.predict_proba(Pool(X_test_ivf, cat_features=cat_idx_ivf))[:, 1] / N_IVF

    # LightGBM
    lb = lgb.LGBMClassifier(**lgb_params)
    lb.fit(X_lgb_ivf.iloc[tr_idx], y_tr,
           eval_set=[(X_lgb_ivf.iloc[val_idx], y_val)],
           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_ivf[val_idx]  = lb.predict_proba(X_lgb_ivf.iloc[val_idx])[:, 1]
    test_lgb_ivf         += lb.predict_proba(X_test_lgb_ivf)[:, 1] / N_IVF

    cat_auc = roc_auc_score(y_val, oof_cat_ivf[val_idx])
    lgb_auc = roc_auc_score(y_val, oof_lgb_ivf[val_idx])
    print(f'[IVF] Fold {fold+1} | CatBoost: {cat_auc:.5f} | LightGBM: {lgb_auc:.5f}')

# IVF 최적 가중치 탐색
best_w_ivf, best_auc_ivf = 0.5, 0.0
for w in np.arange(0.0, 1.05, 0.05):
    auc = roc_auc_score(y_ivf, oof_cat_ivf * w + oof_lgb_ivf * (1-w))
    if auc > best_auc_ivf:
        best_auc_ivf, best_w_ivf = auc, w

oof_ivf   = oof_cat_ivf * best_w_ivf + oof_lgb_ivf * (1 - best_w_ivf)
test_pred_ivf = test_cat_ivf * best_w_ivf + test_lgb_ivf * (1 - best_w_ivf)

print(f'\n✅ IVF OOF AUC: {best_auc_ivf:.5f}  (w_cat={best_w_ivf:.2f}, w_lgb={1-best_w_ivf:.2f})')

In [ ]:
# DI 모델 - CatBoost 단독
N_DI = 3  # DI는 샘플 적으므로 3-fold
skf_di = StratifiedKFold(n_splits=N_DI, shuffle=True, random_state=SEED)

# DI 모델: 단순하게 (과적합 방지)
di_cat_params = {
    'iterations': 500, 'learning_rate': 0.05,
    'depth': 5,           # 얕게
    'l2_leaf_reg': 5.0,   # 강한 정규화
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'scale_pos_weight': spw_di, 'random_seed': SEED,
    'early_stopping_rounds': 30, 'verbose': False, 'use_best_model': True,
}

oof_di   = np.zeros(len(X_di))
test_pred_di = np.zeros(len(X_test_di))
di_fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf_di.split(X_di, y_di)):
    X_tr  = X_di.iloc[tr_idx].reset_index(drop=True)
    X_val = X_di.iloc[val_idx].reset_index(drop=True)
    y_tr  = y_di.iloc[tr_idx].reset_index(drop=True)
    y_val = y_di.iloc[val_idx].reset_index(drop=True)

    cb = CatBoostClassifier(**di_cat_params)
    cb.fit(Pool(X_tr, y_tr, cat_features=cat_idx_di),
           eval_set=Pool(X_val, y_val, cat_features=cat_idx_di))

    val_pred = cb.predict_proba(Pool(X_val, cat_features=cat_idx_di))[:, 1]
    fold_auc = roc_auc_score(y_val, val_pred)

    oof_di[val_idx]   = val_pred
    test_pred_di     += cb.predict_proba(Pool(X_test_di, cat_features=cat_idx_di))[:, 1] / N_DI
    di_fold_scores.append(fold_auc)
    print(f'[DI] Fold {fold+1} | AUC: {fold_auc:.5f}')

di_auc = roc_auc_score(y_di, oof_di)
print(f'\n✅ DI OOF AUC: {di_auc:.5f} ± {np.std(di_fold_scores):.5f}')
print('※ DI 샘플 수 적어 fold 간 편차 클 수 있음')

In [ ]:
# 전체 OOF AUC 계산 + 비교
# IVF/DI OOF를 원래 train 순서로 재조합
oof_split = np.zeros(len(train))
ivf_mask_train = (train['시술 유형'] == 'IVF').values
di_mask_train  = ~ivf_mask_train

oof_split[ivf_mask_train] = oof_ivf
oof_split[di_mask_train]  = oof_di

total_auc = roc_auc_score(train[TARGET], oof_split)

print('=== 전체 OOF AUC 비교 ===')
print(f'  이전 앙상블 (통합 모델) : 0.74041')
print(f'  IVF/DI 분리 모델       : {total_auc:.5f}')
print(f'  개선폭                 : {total_auc - 0.74041:+.5f}')
print()
print(f'  IVF 단독 OOF AUC : {best_auc_ivf:.5f}')
print(f'  DI  단독 OOF AUC : {di_auc:.5f}')

In [ ]:
# 제출 파일 생성
# test 예측값을 원래 test 순서로 재조합
test_pred_split = np.zeros(len(test))
ivf_mask_test = (test['시술 유형'] == 'IVF').values
di_mask_test  = ~ivf_mask_test

test_pred_split[ivf_mask_test] = test_pred_ivf
test_pred_split[di_mask_test]  = test_pred_di

sample_sub = pd.read_csv('/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw/sample_submission.csv') 
submission = sample_sub.copy()
submission[sample_sub.columns[1]] = test_pred_split

submission.to_csv('submission_ivfdi_split_model.csv', index=False, encoding='utf-8-sig')

print('저장 완료: submission_ivfdi_split_model.csv')
print(f'\n최종 요약')
print(f'  IVF/DI 분리 OOF AUC : {total_auc:.5f}')
print(f'  이전 최고 (앙상블)   : 0.74041')
print(f'  개선폭               : {total_auc - 0.74041:+.5f}')
display(submission.head())